In [1]:
import pandas as pd

df = pd.read_excel('/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/ioki/ml-service_now_tagging/data/untagged/1-7-Dec-merged.xlsx')  

In [2]:
# Rename all columns: lowercase and replace spaces with underscores
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

In [3]:
# delete all columns wiht less than 3200 entires
df = df.dropna(thresh=2300, axis=1)
df = df.drop(['updated', 'reporter','updated_by', 'status','reassignment_count','child_incidents','updated_by'], axis=1)


In [4]:
pd.set_option('display.max_columns', None)
df.head()

,number,opened,priority,service,title,description,category,assignment_group,impact,urgency
0,INC0133168,2025-11-30 10:03:13,4 - Low,Broad- YouSeemail - Prod,Support - YouSee mail/Issue with Mit YouSee/Ca...,When the Customer tries to log in to his email...,Application,YouSee Mail Validation,Minor - Errors affecting a group or a few cust...,Low - Total loss or degradation of a non-busin...
1,INC0133171,2025-11-30 10:20:22,4 - Low,TV-Video On Demand (VOD) - PROD,Support - TV/Video on demand/Streamer or Audio,Customer experiences that when they get up at ...,Application,TV-Video On Demand (VOD) – Validation,Minor - Errors affecting a group or a few cust...,Low - Total loss or degradation of a non-busin...
2,INC0133177,2025-11-30 10:53:51,4 - Low,TV -Misc- Prod,Support - TV/Other/Bland Selv,"After the Customer got a new TV, the Customer'...",Application,TV-Misc – Validation,Minor - Errors affecting a group or a few cust...,Low - Total loss or degradation of a non-busin...
3,INC0133167,2025-11-30 09:47:44,4 - Low,Dawn Agent UI - PROD,Support - Dawn/With Customer context/Other,It is listed as an ongoing change\n\nCategory:...,Application,Nuuday Front Desk,Minor - Errors affecting a group or a few cust...,Low - Total loss or degradation of a non-busin...
4,INC0133169,2025-11-30 10:05:27,4 - Low,Dawn Agent UI - PROD,Support - Dawn/With Customer context/Customer ...,"The Customer has changed address, applied CPR ...",Application,YAC-Resolution,Minor - Errors affecting a group or a few cust...,Low - Total loss or degradation of a non-busin...


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2388 entries, 0 to 2387
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   number            2388 non-null   str           
 1   opened            2388 non-null   datetime64[us]
 2   priority          2388 non-null   str           
 3   service           2388 non-null   str           
 4   title             2388 non-null   str           
 5   description       2388 non-null   str           
 6   category          2388 non-null   str           
 7   assignment_group  2388 non-null   str           
 8   impact            2385 non-null   str           
 9   urgency           2387 non-null   str           
dtypes: datetime64[us](1), str(9)
memory usage: 186.7 KB


# EDA for Title and Description
Analyze text length, named entities, and frequent words in the 'title' and 'description' columns.

In [6]:
# Calculate text length for title and description
df['title_length'] = df['title'].astype(str).apply(len)
df['description_length'] = df['description'].astype(str).apply(len)
print('Title length stats:')
print(df['title_length'].describe())
print('\nDescription length stats:')
print(df['description_length'].describe())

Title length stats:
count    2388.000000
mean       58.616834
std        24.789946
min         7.000000
25%        38.000000
50%        51.000000
75%        82.000000
max       127.000000
Name: title_length, dtype: float64

Description length stats:
count    2388.000000
mean      564.406616
std       292.257291
min         2.000000
25%       366.000000
50%       530.000000
75%       698.250000
max      3235.000000
Name: description_length, dtype: float64


In [7]:
# Most common words in title and description
from collections import Counter
import re

def get_words(text_series):
    words = []
    for text in text_series.dropna():
        words += re.findall(r'\w+', text.lower())
    return words

title_words = get_words(df['title'])
description_words = get_words(df['description'])

print('Top 20 words in title:')
print(Counter(title_words).most_common(20))
print('\nTop 20 words in description:')
print(Counter(description_words).most_common(20))

Top 20 words in title:
[('support', 1976), ('dawn', 1264), ('with', 1027), ('customer', 968), ('context', 738), ('other', 658), ('order', 611), ('yousee', 570), ('not', 451), ('and', 448), ('quotation', 412), ('orders', 393), ('quotes', 385), ('cannot', 381), ('or', 293), ('tv', 286), ('mail', 267), ('completed', 266), ('my', 248), ('issue', 234)]

Top 20 words in description:
[('the', 9745), ('customer', 6984), ('to', 5035), ('is', 3405), ('and', 3363), ('not', 2603), ('has', 2546), ('it', 2461), ('in', 2330), ('order', 2326), ('with', 2255), ('a', 1993), ('category', 1975), ('on', 1875), ('dawn', 1705), ('that', 1638), ('error', 1631), ('app', 1626), ('but', 1577), ('cannot', 1512)]


In [8]:
# NER tagging for company-specific names using spaCy
import spacy
nlp = spacy.load('en_core_web_sm')

def extract_orgs(text_series):
    orgs = []
    for text in text_series.dropna():
        doc = nlp(text)
        orgs += [ent.text for ent in doc.ents if ent.label_ == 'ORG']
    return Counter(orgs)

title_orgs = extract_orgs(df['title'])
description_orgs = extract_orgs(df['description'])

print('Top 10 organizations in title:')
print(title_orgs.most_common(10))
print('\nTop 10 organizations in description:')
print(description_orgs.most_common(10))

Top 10 organizations in title:
[('Support - Smartwatch/Add', 23), ('Icon', 20), ('Support - Music/Missing', 15), ('Order Command', 11), ('ACS', 10), ('NP', 9), ('MBB', 9), ('IP Ex', 8), ('Support - Smartwatch/Other', 8), ('Stuck Pending', 8)]

Top 10 organizations in description:
[('Customer', 2430), ('YouSee', 435), ('Netcracker', 108), ('DAWN', 92), ('Quote', 64), ('SIM', 54), ('YouSee Music', 41), ('OCH', 39), ('MobilePay', 37), ('TDC', 34)]


In [ ]:
# Add stopwords and matplotlib imports
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

In [ ]:
# Most common words in title and description, removing stopwords
from collections import Counter
import re

def get_words(text_series):
    words = []
    for text in text_series.dropna():
        words += re.findall(r'\w+', text.lower())
    return words

def remove_stopwords(words):
    return [w for w in words if w not in stop_words]

title_words = get_words(df['title'])
description_words = get_words(df['description'])

title_words_nostop = remove_stopwords(title_words)
description_words_nostop = remove_stopwords(description_words)

title_counter = Counter(title_words_nostop)
description_counter = Counter(description_words_nostop)

print('Top 20 words in title (no stopwords):')
print(title_counter.most_common(20))
print('\nTop 20 words in description (no stopwords):')
print(description_counter.most_common(20))

In [ ]:
# Visualize text length distributions
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
df['title_length'].hist(bins=30)
plt.title('Title Length Distribution')
plt.xlabel('Length')
plt.ylabel('Frequency')

plt.subplot(1,2,2)
df['description_length'].hist(bins=30)
plt.title('Description Length Distribution')
plt.xlabel('Length')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize top 20 words in title and description (no stopwords)
fig, axes = plt.subplots(1, 2, figsize=(16,6))
title_common = title_counter.most_common(20)
desc_common = description_counter.most_common(20)

axes[0].barh([w for w, _ in reversed(title_common)], [c for _, c in reversed(title_common)])
axes[0].set_title('Top 20 Title Words (No Stopwords)')
axes[0].set_xlabel('Count')

axes[1].barh([w for w, _ in reversed(desc_common)], [c for _, c in reversed(desc_common)])
axes[1].set_title('Top 20 Description Words (No Stopwords)')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()